# Train LoRA của mình từ Qwen3.5-9B base
Upload `repo.zip` (checkout có script này) và `train.zip` vào **MyDrive/datamate/**. Chọn GPU **A100 / H100 / G4** và Python >=3.12; cần RAM cao và đủ dung lượng Drive cho model (~20 GB), hai môi trường tạm và checkpoint. Xem `docs/colab-training.md` trước khi chạy. Notebook tự trả runtime sau ô cuối, kể cả khi lỗi; chạy lại cùng OUTPUT để tiếp tục. Không tải hoặc dùng adapter công bố.


In [ ]:
from google.colab import drive, runtime
from pathlib import Path

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/datamate")


In [ ]:
# Chỉ sửa ô này. Giữ nguyên tham số + OUTPUT khi resume.
REPO_ZIP = DRIVE / "repo.zip"
TRAIN_ZIP = DRIVE / "train.zip"
PUBLIC_ZIP = DRIVE / "public_test.zip"  # cho STAGES = "public"
OUTPUT = DRIVE / "from_base"
STAGES = "all"  # "setup,preflight" đo tốc độ; "all" sinh dữ liệu + train; "setup,public" chạy 1000 câu public
GPU = "auto"   # tự nhận A100 (bf16), H100/G4 (fp8 nếu compute capability >=8.9)
MAX_LEN = 24576
EPOCHS = 1
GRAD_ACCUM = 8
SAVE_STEPS = 10
PUBLIC_THINK_K = 4  # 0 = nhanh hơn, yếu hơn
MAX_STEPS = -1  # thử 3 bước: dùng OUTPUT khác; không đổi tham số của run đang resume


In [ ]:
import os
import shutil
import signal
import subprocess
import sys
import zipfile

# Mọi artifact bền vững và log ở Drive; code/venv cục bộ được dựng lại khi mất phiên.
log_path = OUTPUT / "colab.log"

def unpack(src, dest):
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src) as archive:
        for member in archive.infolist():
            if not (dest / member.filename).resolve().is_relative_to(dest.resolve()):
                raise ValueError("Unsafe path in zip")
        archive.extractall(dest)

try:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        try:
            code_dir = Path("/content/datamate-from-base-code")
            if code_dir.exists():
                shutil.rmtree(code_dir)  # refresh code from the uploaded zip, never clone or fetch LFS
            unpack(REPO_ZIP, code_dir)
            candidates = list(code_dir.rglob("scripts/colab_train.py"))
            if len(candidates) != 1:
                raise RuntimeError("repo.zip must contain exactly one scripts/colab_train.py")
            root = candidates[0].parents[1]
            inputs = OUTPUT / "input"
            ready = inputs / ".train-extracted"
            if not ready.exists():
                unpack(TRAIN_ZIP, inputs)
                candidates = [p.parent for p in inputs.rglob("questions.jsonl")
                              if p.parent.name == "train"]
                # Also accept train.zip with questions.jsonl directly at the zip root.
                if not candidates and (inputs / "questions.jsonl").exists():
                    staging = inputs / ".flat-train"
                    staging.mkdir(parents=True, exist_ok=True)
                    for p in list(inputs.iterdir()):
                        if p != staging:
                            shutil.move(str(p), str(staging / p.name))
                    target = inputs / "train"
                    staging.rename(target)
                    candidates = [target]
                if len(candidates) != 1:
                    raise RuntimeError("train.zip must contain one train split (questions.jsonl, rulebook.md, data/)")
                train = candidates[0]
                for name in ("questions.jsonl", "answers.jsonl", "rulebook.md", "data"):
                    if not (train / name).exists():
                        raise RuntimeError(f"Missing train/{name}")
                ready.write_text(str(train), encoding="utf-8")
            train = Path(ready.read_text(encoding="utf-8"))
            if "public" in STAGES and not (inputs / "public_test" / "questions.jsonl").exists():
                unpack(PUBLIC_ZIP, inputs)  # public_test.zip chứa thư mục public_test/
            cmd = [sys.executable, "-u", str(root / "scripts/colab_train.py"),
                   "--output", str(OUTPUT), "--data", str(train.parent), "--stage", STAGES,
                   "--gpu", GPU, "--max-len", str(MAX_LEN), "--epochs", str(EPOCHS),
                   "--grad-accum", str(GRAD_ACCUM), "--save-steps", str(SAVE_STEPS),
                   "--max-steps", str(MAX_STEPS), "--public-think-k", str(PUBLIC_THINK_K)]
            process = subprocess.Popen(cmd, cwd=root, env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                                       start_new_session=True)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                code = process.wait()
            finally:
                if process.poll() is None:
                    os.killpg(process.pid, signal.SIGTERM)
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        os.killpg(process.pid, signal.SIGKILL)
                        process.wait()
            if code:
                raise RuntimeError(f"Flow exited {code}; resume with the same OUTPUT after checking {log_path}")
        except BaseException as exc:
            log.write(f"\nSTOP: {type(exc).__name__}: {exc}\n")
            log.flush()
            raise
finally:
    print(f"Log: {log_path}. Trả runtime để dừng tính compute units.", flush=True)
    runtime.unassign()
